# M12 — Final Backbone Selection — OWMTL Project**Model ID:** M12 &nbsp;&nbsp;|&nbsp;&nbsp; **Member:** A (Backbone Engineering) &nbsp;&nbsp;|&nbsp;&nbsp; **Author:** Asif**Phase:** 3 (Week 6) &nbsp;&nbsp;|&nbsp;&nbsp; **Requires:** M2, M3, M4 (full results from all three)---### ObjectiveM12 is **the decision, documented** — pick the winning encoder and freeze it as the shared backboneevery downstream model builds on. `Model_Training_Reference.md:62` calls it *"Member A's wholedefense answer"*, and `:244` defines the deliverable as **one frozen checkpoint + a short writtenjustification**.**This notebook trains nothing.** It loads the completed `results_M*.json` files, applies a decisionrule that is stated *before* the numbers are read, verifies the winning checkpoint actuallyreproduces its reported metrics, and emits the justification document programmatically so it cannotdrift from the data it claims to be based on.---### Why this is being run now`Model_Training_Reference.md:114` requires **M2, M3, and M4** as inputs. Until M2 and M3 completed(2026-07-30), only M4 existed, so the earlier pass at M12 had to decide between two models — and itselected **M1**, which `:144` explicitly designates a throwaway:> *"treat this as a throwaway/reference checkpoint — the real reported numbers come from M2 (CNN> baseline, tuned)."*M1 is therefore **not an eligible candidate**, and this notebook excludes it from the decision whilestill reporting it for reference. This is the first time the required evidence has existed.---### The deliverables this produces| File | Purpose ||---|---|| `results_M12.json` | §4-schema record of the decision, inheriting the winner's metrics || `M12_backbone_justification.md` | The written defense (§244) — generated from the data || `backbone_comparison.png` | Four-way metric comparison || `efficiency_vs_performance.png` | The accuracy-vs-cost tradeoff the decision rests on || `decision_audit.json` | Every rule step and its outcome, for reproducibility |---### Running thisUnlike M2/M3 this needs **no GPU and no Colab** — it is pure analysis over JSON files, so it runslocally in seconds from the repo root. Checkpoint verification (Section 6) is the one optional stepthat needs the dataset and a GPU; it self-skips cleanly when they are absent.

---## Section 1 — Environment Setup & Path DiscoveryLocates the repo, the candidate results files, and the output directory. Works from the repo root,from inside `Asif's/M12/`, on Colab with Drive mounted, or on Kaggle.

In [ ]:
# ============================================================
# CELL 0 — ENVIRONMENT & PATH DISCOVERY
# ============================================================
import os, sys, glob, platform

print("=" * 72)
print("M12 BACKBONE SELECTION — ENVIRONMENT")
print("=" * 72)
print(f"Python   : {sys.version.split()[0]}  ({platform.platform()})")

IN_COLAB = "google.colab" in sys.modules
DRIVE_MOUNTED = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_MOUNTED = True
        print("Google Drive mounted at /content/drive")
    except Exception as e:
        print(f"Drive mount skipped ({e})")


def find_repo_root(start=None):
    """
    Walk upward looking for the OWMTL repo root (identified by the two protocol docs).
    Returns None when run outside the repo, e.g. a bare Colab session.
    """
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        markers = [os.path.join(d, "Model_Training_Reference.md"),
                   os.path.join(d, "Model_Training_Protocol.md")]
        if all(os.path.exists(m) for m in markers):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None


REPO_ROOT = find_repo_root()
print(f"Repo root: {REPO_ROOT or 'not found (running outside the repo — that is fine)'}")

# Output directory: prefer the repo so the deliverables land where the team looks for them.
if REPO_ROOT:
    OUT_DIR = os.path.join(REPO_ROOT, "Asif's", "M12")
elif IN_COLAB and DRIVE_MOUNTED:
    OUT_DIR = "/content/drive/MyDrive/OWMTL/M12"
elif IN_COLAB:
    OUT_DIR = "/content/OWMTL/M12"
elif os.path.exists("/kaggle/working"):
    OUT_DIR = "/kaggle/working"
else:
    OUT_DIR = "./outputs_M12"

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Output   : {OUT_DIR}")
print("=" * 72)

---## Section 2 — Configuration`CFG` holds the candidate roster, the search paths, and the decision-rule parameters. The rule'sparameters live here — separate from the numbers — so that changing the rule is a visible,reviewable edit rather than something buried in the analysis.

In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION
# ============================================================
CFG = {
    # ── Eligibility (Model_Training_Reference.md:114) ──
    # M12 "requires M2, M3, M4". M1 is excluded by :144, which designates it a
    # throwaway/reference checkpoint whose numbers are explicitly superseded by M2.
    "candidates": ["M2", "M3", "M4"],
    "reference_only": ["M1"],

    # ── Decision-rule parameters (see Section 4) ──
    "primary_metric": "icbhi_score",
    # Two candidates count as statistically tied when their primary-metric gap is
    # smaller than this multiple of the leader's cross-validation std.
    "tie_threshold_cv_multiples": 1.0,
    # Fallback spread used when a candidate reports no CV std of its own.
    "default_cv_std": 0.0129,
    "secondary_metrics": ["accuracy", "f1_macro"],

    # ── Verification (Section 6) ──
    "verify_checkpoint": True,     # self-skips when dataset/checkpoint/GPU are unavailable

    # ── Identity ──
    "model_id": "M12",
    "model_name": "Backbone Selection (Final)",
    "member": "A",
    "member_name": "Asif",
    "out_dir": OUT_DIR,
}

# ------------------------------------------------------------
# Where to look for each candidate's results JSON.
# Covers the repo layout, the Drive layout M2/M3 wrote to, and plain uploads.
# ------------------------------------------------------------
def results_search_paths(model_id):
    names = [f"results_{model_id}.json"]
    roots = []
    if REPO_ROOT:
        for member in ["Asif's", "Barshon's", "Farhana's", "Sami's"]:
            roots += [
                os.path.join(REPO_ROOT, member, model_id),
                os.path.join(REPO_ROOT, member, model_id, "results"),
                os.path.join(REPO_ROOT, member, f"{model_id} [UPDATED]"),
            ]
    roots += [
        f"/content/drive/MyDrive/OWMTL/{model_id}/results",
        f"/content/drive/MyDrive/OWMTL/{model_id}",
        "/content", "/kaggle/input", ".",
    ]
    return [os.path.join(r, n) for r in roots for n in names]


print("M12 CONFIGURATION")
print("-" * 72)
print(f"  Eligible candidates : {CFG['candidates']}")
print(f"  Reference only      : {CFG['reference_only']}  (excluded from the decision)")
print(f"  Primary metric      : {CFG['primary_metric']}")
print(f"  Tie threshold       : {CFG['tie_threshold_cv_multiples']} x CV std")
print(f"  Verify checkpoint   : {CFG['verify_checkpoint']}")
print("-" * 72)

In [ ]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================
import json, math, shutil, datetime, warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_style("whitegrid")
    HAS_SNS = True
except ImportError:
    HAS_SNS = False
    print("seaborn not installed — plots will use plain matplotlib styling.")


class NumpyEncoder(json.JSONEncoder):
    """Serialise numpy scalars/arrays (§11.A safety for every json.dump here)."""

    def default(self, o):
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.bool_):
            return bool(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return super().default(o)


print("Imports OK.")

---## Section 3 — Load Candidate ResultsEach candidate is loaded from its `results_M*.json` and **validated against the §4 schema** beforeit is allowed into the decision. A file that is missing required blocks is reported rather thansilently trusted.**On M4:** its `results_M4.json` was never committed — Barshon's M4 notebook wrote it to hispersonal Google Drive, and only the Drive *link* is in the repo(`Barshon's/M4/Best_model_pth_file Link.txt`). The values below are therefore transcribed from theexecuted output of `Barshon's/M4/M4_ast_backbone.ipynb`, which is committed and reproducible. If thereal JSON is ever added to the repo or mounted from Drive, this notebook picks it up automaticallyand the transcription is discarded.

In [ ]:
# ============================================================
# CELL 3 — LOAD & VALIDATE CANDIDATE RESULTS
# ============================================================
#
# RECORDED FALLBACKS
# ------------------
# Used only when a candidate's results_M*.json cannot be found on disk.
# M1/M2/M3 are committed, so their fallbacks are belt-and-braces.
# M4's is the real path in practice — see the markdown above for provenance.

RECORDED = {
    "M1": {
        "model_name": "Provisional CNN Backbone", "architecture": "2D_CNN_4Block",
        "accuracy": 0.5407, "precision_macro": 0.5295, "recall_macro": 0.5801,
        "f1_macro": 0.4844, "specificity_macro": 0.8561, "icbhi_score": 0.7181,
        "total_params": 421_732, "model_size_mb": 4.85,
        "inference_time_ms_per_sample": 1.85, "best_epoch": 29, "cv_std": None,
        "source": "recorded from Barshon's/M1/results_M1.json",
    },
    "M2": {
        "model_name": "CNN Baseline (Tuned, Final)", "architecture": "2D_CNN_5Block_w48_do0.4",
        "accuracy": 0.6138, "precision_macro": 0.5097, "recall_macro": 0.5817,
        "f1_macro": 0.5238, "specificity_macro": 0.8638, "icbhi_score": 0.7227,
        "total_params": 3_627_476, "model_size_mb": 13.86,
        "inference_time_ms_per_sample": 2.937, "best_epoch": 19, "cv_std": 0.0129,
        "source": "recorded from Asif's/M2/results_M2.json",
    },
    "M3": {
        "model_name": "MobileNet/DenseNet Lightweight Backbone", "architecture": "mobilenet_v2",
        "accuracy": 0.5915, "precision_macro": 0.4848, "recall_macro": 0.5431,
        "f1_macro": 0.4904, "specificity_macro": 0.8537, "icbhi_score": 0.6984,
        "total_params": 2_228_996, "model_size_mb": 8.74,
        "inference_time_ms_per_sample": 5.42, "best_epoch": 4, "cv_std": 0.0068,
        "source": "recorded from Asif's/M3/results_M3.json",
    },
    "M4": {
        "model_name": "AST (Audio Spectrogram Transformer)", "architecture": "AST_pretrained",
        "accuracy": 0.5528, "precision_macro": 0.5347, "recall_macro": 0.4385,
        "f1_macro": 0.4109, "specificity_macro": 0.8334, "icbhi_score": 0.6359,
        "total_params": 86_385_668,
        # NOTE: the 988.85 MB figure printed by M4's handoff cell is the *checkpoint file*
        # size, which includes optimiser state -- not the state_dict the protocol asks for
        # (§3: "Save state_dict() to temp file, check file size"). At fp32 the state_dict
        # is 86,385,668 x 4 bytes = 329.5 MB. Using the inflated number would overstate
        # AST's disadvantage by ~3x, so the state_dict estimate is used here and the
        # discrepancy is recorded in the audit.
        "model_size_mb": round(86_385_668 * 4 / (1024 ** 2), 2),
        "model_size_mb_note": ("state_dict estimate at fp32; M4's handoff cell reported "
                               "988.85 MB for the full checkpoint incl. optimiser state"),
        "inference_time_ms_per_sample": 86.74, "best_epoch": 10, "cv_std": None,
        "source": ("TRANSCRIBED from executed output of Barshon's/M4/M4_ast_backbone.ipynb "
                   "(results_M4.json is not in the repo -- it lives in Barshon's Drive)"),
    },
}

# §4 blocks a compliant results file must contain
REQUIRED_BLOCKS = ["meta", "config", "efficiency", "best_epoch", "best_metrics"]
REQUIRED_METRICS = ["accuracy", "precision_macro", "recall_macro", "f1_macro",
                    "specificity_macro", "icbhi_score"]


def validate_schema(payload, model_id):
    """Return a list of §4 schema problems (empty list == compliant)."""
    problems = []
    for blk in REQUIRED_BLOCKS:
        if blk not in payload:
            problems.append(f"missing §4 block '{blk}'")
    for m in REQUIRED_METRICS:
        if m not in payload.get("best_metrics", {}):
            problems.append(f"missing §3 metric 'best_metrics.{m}'")
    if "ablation" not in payload:
        problems.append("missing §4.1 'ablation' block")
    return problems


def load_candidate(model_id):
    """Load one candidate from disk, else fall back to the recorded values."""
    for path in results_search_paths(model_id):
        if not os.path.exists(path):
            continue
        try:
            payload = json.load(open(path))
        except Exception as e:
            print(f"  {model_id}: found {path} but could not parse it ({e})")
            continue

        problems = validate_schema(payload, model_id)
        bm, eff = payload.get("best_metrics", {}), payload.get("efficiency", {})
        rec = {
            "model_name": payload.get("meta", {}).get("model_name", model_id),
            "architecture": payload.get("config", {}).get("architecture", "?"),
            **{m: bm.get(m) for m in REQUIRED_METRICS},
            "total_params": eff.get("total_params"),
            "model_size_mb": eff.get("model_size_mb"),
            "inference_time_ms_per_sample": eff.get("inference_time_ms_per_sample"),
            "best_epoch": payload.get("best_epoch", {}).get("epoch"),
            "cv_std": (payload.get("hp_sweep", {}).get("all_results") or [{}])[0].get("std_icbhi"),
            "source": f"loaded from {path}",
            "schema_problems": problems,
            "_raw": payload,
        }
        status = "OK" if not problems else f"{len(problems)} schema issue(s)"
        print(f"  {model_id}: loaded from {os.path.relpath(path, REPO_ROOT or os.getcwd())}  [{status}]")
        for p in problems:
            print(f"        - {p}")
        return rec

    rec = dict(RECORDED[model_id])
    rec["schema_problems"] = ["results JSON not found on disk — using recorded values"]
    print(f"  {model_id}: NOT FOUND on disk — using recorded values")
    print(f"        source: {rec['source']}")
    return rec


print("Loading candidate results")
print("-" * 72)
ALL_MODELS = {}
for mid in CFG["reference_only"] + CFG["candidates"]:
    ALL_MODELS[mid] = load_candidate(mid)
print("-" * 72)

# Every eligible candidate must at minimum carry the primary metric.
missing = [m for m in CFG["candidates"]
           if ALL_MODELS[m].get(CFG["primary_metric"]) is None]
assert not missing, (
    f"Cannot run M12: candidate(s) {missing} have no {CFG['primary_metric']}. "
    f"M12 requires complete results for M2, M3 and M4 (Model_Training_Reference.md:114).")
print(f"All {len(CFG['candidates'])} eligible candidates have a primary metric. Ready to decide.")

In [ ]:
# ============================================================
# CELL 4 — THE COMPARISON TABLE
# ============================================================
# All rows share: ICBHI 2017, official patient-independent 60/40 split, cycle-level,
# 4-class sound event, inverse-frequency class-weighted CrossEntropyLoss, no augmentation,
# seed 42. That shared setup is what makes the comparison free of the loss-formulation
# confound Model_Training_Reference.md:243 warns about.

ORDER = CFG["reference_only"] + CFG["candidates"]

def fmt(v, spec=".4f", na="n/a"):
    return na if v is None else format(v, spec)

print("=" * 118)
print("M12 BACKBONE COMPARISON — ICBHI 2017, official 60/40, class-weighted CE, no augmentation")
print("=" * 118)
hdr = (f"{'Model':<8}{'Architecture':<26}{'Acc':>8}{'Prec':>8}{'Se':>8}{'Sp':>8}"
       f"{'F1':>8}{'ICBHI':>9}{'Params':>13}{'Size MB':>10}{'ms/samp':>9}{'':>4}")
print(hdr)
print("-" * 118)
for mid in ORDER:
    r = ALL_MODELS[mid]
    flag = "(ref)" if mid in CFG["reference_only"] else ""
    print(f"{mid:<8}{str(r['architecture'])[:25]:<26}"
          f"{fmt(r['accuracy']):>8}{fmt(r['precision_macro']):>8}{fmt(r['recall_macro']):>8}"
          f"{fmt(r['specificity_macro']):>8}{fmt(r['f1_macro']):>8}{fmt(r['icbhi_score']):>9}"
          f"{(r['total_params'] or 0):>13,}{fmt(r['model_size_mb'], '.2f'):>10}"
          f"{fmt(r['inference_time_ms_per_sample'], '.2f'):>9}{flag:>6}")
print("=" * 118)
print("(ref) = reported for context only; excluded from the decision per "
       "Model_Training_Reference.md:144")

# Provenance matters for a decision document — say where every row came from.
print("\nProvenance:")
for mid in ORDER:
    print(f"  {mid}: {ALL_MODELS[mid]['source']}")

flagged = {m: r["schema_problems"] for m, r in ALL_MODELS.items() if r.get("schema_problems")}
if flagged:
    print("\nSchema / availability warnings:")
    for m, probs in flagged.items():
        for p in probs:
            print(f"  {m}: {p}")

---## Section 4 — The Decision Rule`Model_Training_Reference.md:243` is explicit that this must be *"a deliberate, documentedefficiency-vs-accuracy tradeoff decision, not just 'highest accuracy wins'"*, because Member C'scompression workstream depends on the chosen teacher being a reasonable size.The rule is therefore stated as executable code **before** it is applied, so the decision cannot bereverse-engineered to fit a preferred answer:| Step | Rule ||---|---|| **1. Eligibility** | Candidates are M2, M3, M4 (`:114`). M1 is excluded (`:144` — throwaway checkpoint, superseded by M2). || **2. Rank** | Order by ICBHI score, the field-standard primary metric for this dataset. || **3. Separation** | The leader wins outright **only if** its margin over the runner-up exceeds the leader's cross-validation std. Otherwise the two are *tied* and go to step 4. || **4. Efficiency tiebreak** | Among tied candidates, prefer the smaller/faster model (params → size → latency), per `:243`. || **5. Sanity gate** | The winner must not be beaten by another candidate on **both** accuracy and macro-F1. Guards against an ICBHI-only artifact. |Step 3 is what stops this being a rubber-stamp of the top row. The CV std comes from each model's ownpatient-grouped sweep (`M2_hp_sweep.json`, `M3_hp_sweep.json`) — a real measurement of how much thatarchitecture's score moves between patient folds, not an assumed tolerance.

In [ ]:
# ============================================================
# CELL 5 — THE DECISION RULE (defined before it is applied)
# ============================================================

def decide_backbone(models, cfg):
    """
    Apply the five-step rule from Section 4.

    Returns (winner_id, audit) where `audit` records every step's inputs and
    outcome so the decision is reproducible from the JSON alone.
    """
    metric = cfg["primary_metric"]
    audit = {"rule_version": "1.0", "primary_metric": metric, "steps": []}

    # ── Step 1: eligibility ──
    eligible = list(cfg["candidates"])
    audit["steps"].append({
        "step": 1, "name": "eligibility",
        "eligible": eligible, "excluded": cfg["reference_only"],
        "rationale": ("Model_Training_Reference.md:114 requires M2, M3, M4. "
                      "M1 excluded by :144 (throwaway checkpoint superseded by M2)."),
    })

    # ── Step 2: rank by primary metric ──
    ranked = sorted(eligible, key=lambda m: models[m][metric], reverse=True)
    audit["steps"].append({
        "step": 2, "name": "rank_by_primary_metric",
        "ranking": [{"model": m, metric: models[m][metric]} for m in ranked],
    })

    leader, runner_up = ranked[0], ranked[1]
    margin = models[leader][metric] - models[runner_up][metric]
    cv_std = models[leader].get("cv_std") or cfg["default_cv_std"]
    threshold = cfg["tie_threshold_cv_multiples"] * cv_std
    separated = margin > threshold

    # ── Step 3: statistical separation ──
    audit["steps"].append({
        "step": 3, "name": "statistical_separation",
        "leader": leader, "runner_up": runner_up,
        "margin": round(float(margin), 4),
        "leader_cv_std": round(float(cv_std), 4),
        "threshold": round(float(threshold), 4),
        "separated": bool(separated),
        "interpretation": (
            f"{leader} leads {runner_up} by {margin:.4f}, which "
            f"{'exceeds' if separated else 'does NOT exceed'} the {threshold:.4f} "
            f"cross-validation tolerance."),
    })

    if separated:
        tied, winner = [leader], leader
        audit["steps"].append({
            "step": 4, "name": "efficiency_tiebreak", "applied": False,
            "rationale": "Leader is statistically separated; no tiebreak needed.",
        })
    else:
        tied = [m for m in ranked
                if models[leader][metric] - models[m][metric] <= threshold]
        # Efficiency preference: fewer params, then smaller, then faster.
        winner = sorted(tied, key=lambda m: (
            models[m].get("total_params") or float("inf"),
            models[m].get("model_size_mb") or float("inf"),
            models[m].get("inference_time_ms_per_sample") or float("inf")))[0]
        audit["steps"].append({
            "step": 4, "name": "efficiency_tiebreak", "applied": True,
            "tied_candidates": tied, "selected": winner,
            "rationale": ("Model_Training_Reference.md:243 — among statistically "
                          "indistinguishable candidates prefer the more deployable one, "
                          "since Member C's compression work starts from this teacher."),
        })

    # ── Step 5: sanity gate ──
    dominated_by = [
        m for m in eligible
        if m != winner
        and all((models[m].get(s) or -1) > (models[winner].get(s) or -1)
                for s in cfg["secondary_metrics"])
    ]
    passed = not dominated_by
    audit["steps"].append({
        "step": 5, "name": "sanity_gate",
        "secondary_metrics": cfg["secondary_metrics"],
        "models_beating_winner_on_all_secondaries": dominated_by,
        "passed": bool(passed),
        "rationale": ("Guards against selecting a model that wins on ICBHI alone while "
                      "losing on both accuracy and macro-F1."),
    })

    audit["winner"] = winner
    audit["sanity_gate_passed"] = bool(passed)
    audit["tied_group"] = tied
    return winner, audit


print("Decision rule defined (v1.0, 5 steps). Not yet applied.")

In [ ]:
# ============================================================
# CELL 6 — APPLY THE RULE
# ============================================================
WINNER, AUDIT = decide_backbone(ALL_MODELS, CFG)
w = ALL_MODELS[WINNER]

print("=" * 78)
print("DECISION TRACE")
print("=" * 78)
for step in AUDIT["steps"]:
    print(f"\nSTEP {step['step']} — {step['name']}")
    for k, v in step.items():
        if k in ("step", "name"):
            continue
        if k == "ranking":
            for i, row in enumerate(v, 1):
                print(f"    {i}. {row['model']:<4} {CFG['primary_metric']} = "
                      f"{row[CFG['primary_metric']]:.4f}")
        else:
            print(f"    {k}: {v}")

print("\n" + "=" * 78)
print(f"SELECTED BACKBONE: {WINNER} — {w['model_name']}")
print("=" * 78)
print(f"  Architecture     : {w['architecture']}")
print(f"  ICBHI Score      : {w['icbhi_score']:.4f}")
print(f"  Accuracy         : {w['accuracy']:.4f}")
print(f"  Macro-F1         : {w['f1_macro']:.4f}")
print(f"  Parameters       : {w['total_params']:,}")
print(f"  Model size       : {w['model_size_mb']} MB")
print(f"  Inference        : {w['inference_time_ms_per_sample']} ms/sample")
print(f"  Sanity gate      : {'PASSED' if AUDIT['sanity_gate_passed'] else 'FAILED'}")

if not AUDIT["sanity_gate_passed"]:
    print("\n  WARNING: the sanity gate did not pass. Another candidate beats the winner "
          "on both accuracy and macro-F1. Review before freezing this backbone.")

# How does the winner compare to what the previous M12 pass selected?
prev = "M1"
if prev in ALL_MODELS and WINNER != prev:
    p = ALL_MODELS[prev]
    print(f"\nChange from the earlier M12 pass (which selected {prev}):")
    for k, label in [("icbhi_score", "ICBHI"), ("accuracy", "Accuracy"), ("f1_macro", "Macro-F1")]:
        print(f"  {label:<10}: {p[k]:.4f} -> {w[k]:.4f}   ({w[k] - p[k]:+.4f})")
    print(f"  Size      : {p['model_size_mb']} MB -> {w['model_size_mb']} MB")
    print(f"  {WINNER} dominates {prev} on every reported metric, and unlike {prev} it is an "
          f"eligible candidate under Model_Training_Reference.md:114.")

---### Robustness check — what if M1 were an eligible candidate?M1 is excluded on documentary grounds (`Model_Training_Reference.md:144` designates it a throwawaywhose numbers are superseded by M2). A reviewer will reasonably ask whether that exclusion is doingthe work of the decision. The cell below answers it by re-running the **same rule** with M1promoted to a full candidate.This matters more than it looks: M2 leads M1 by only **0.0046** ICBHI, which is *below* M2's owncross-validation tolerance — so on the primary metric alone the two are statistically tied, theefficiency tiebreak fires, and the smaller model (M1, 422 K params) wins. That is almost certainlyhow the earlier M12 pass arrived at M1.The sanity gate is what stops it: M2 and M3 both beat M1 on accuracy **and** macro-F1, so aselection of M1 fails step 5. The exclusion and the numbers therefore agree — which is thestrongest form this argument can take.

In [ ]:
# ============================================================
# CELL 6b — ROBUSTNESS: M1 PROMOTED TO A FULL CANDIDATE
# ============================================================
# Re-applies the identical rule with M1 eligible, to show the decision does not
# rest on the exclusion alone. Results are recorded in the audit and the
# justification document.

_cf_cfg = dict(CFG)
_cf_cfg["candidates"] = ["M1"] + list(CFG["candidates"])
_cf_cfg["reference_only"] = []

CF_WINNER, CF_AUDIT = decide_backbone(ALL_MODELS, _cf_cfg)
_cf_sep = next(s for s in CF_AUDIT["steps"] if s["name"] == "statistical_separation")
_cf_tie = next(s for s in CF_AUDIT["steps"] if s["name"] == "efficiency_tiebreak")
_cf_san = next(s for s in CF_AUDIT["steps"] if s["name"] == "sanity_gate")

print("=" * 78)
print("ROBUSTNESS CHECK — same rule, M1 promoted to a full candidate")
print("=" * 78)
print(f"  candidates      : {_cf_cfg['candidates']}")
print("  ranking (ICBHI) : " + " > ".join(
    f"{r['model']} {r[CFG['primary_metric']]:.4f}"
    for r in next(s for s in CF_AUDIT["steps"]
                  if s["name"] == "rank_by_primary_metric")["ranking"]))
print(f"  leader margin   : {_cf_sep['leader']} over {_cf_sep['runner_up']} "
      f"by {_cf_sep['margin']:.4f} (tolerance {_cf_sep['threshold']:.4f})")
print(f"  separated?      : {_cf_sep['separated']}")
print(f"  tiebreak fired? : {_cf_tie['applied']}"
      + (f"   tied={_cf_tie.get('tied_candidates')} -> {_cf_tie.get('selected')}"
         if _cf_tie["applied"] else ""))
print(f"  would select    : {CF_WINNER}")
print(f"  sanity gate     : {'PASSED' if _cf_san['passed'] else 'FAILED'}")
if not _cf_san["passed"]:
    print(f"                    {_cf_san['models_beating_winner_on_all_secondaries']} "
          f"beat {CF_WINNER} on BOTH accuracy and macro-F1")
print("-" * 78)

ROBUSTNESS = {
    "question": "Does the decision depend on excluding M1?",
    "counterfactual_candidates": _cf_cfg["candidates"],
    "counterfactual_winner": CF_WINNER,
    "counterfactual_sanity_gate_passed": bool(_cf_san["passed"]),
    "canonical_winner": WINNER,
    "decision_changes": bool(CF_WINNER != WINNER),
    "audit": CF_AUDIT,
}

if CF_WINNER == WINNER:
    ROBUSTNESS["conclusion"] = (
        f"The decision is unchanged ({WINNER}) whether or not M1 is eligible, so it "
        f"does not rest on the exclusion.")
elif not _cf_san["passed"]:
    ROBUSTNESS["conclusion"] = (
        f"Promoting M1 would select {CF_WINNER}, but that selection FAILS the sanity "
        f"gate: {_cf_san['models_beating_winner_on_all_secondaries']} beat it on both "
        f"accuracy and macro-F1. M2 leads M1 by only {_cf_sep['margin']:.4f} ICBHI -- "
        f"inside the {_cf_sep['threshold']:.4f} cross-validation tolerance -- so on the "
        f"primary metric alone they are tied and the efficiency tiebreak prefers the "
        f"smaller M1. This is almost certainly how the earlier M12 pass selected M1. "
        f"The documentary exclusion and the numbers therefore agree: {WINNER} is the "
        f"defensible choice, and the real evidence for it is accuracy and macro-F1, "
        f"not ICBHI score.")
else:
    ROBUSTNESS["conclusion"] = (
        f"WARNING: promoting M1 selects {CF_WINNER} and PASSES the sanity gate. The "
        f"decision depends materially on the exclusion -- review before freezing.")

print(ROBUSTNESS["conclusion"])
print("=" * 78)

---## Section 5 — FiguresTwo plots: the four-way metric comparison, and the efficiency-vs-performance tradeoff that thedecision rule's step 4 would have used had the leader not been statistically separated. The secondis the figure that belongs in the paper — it shows the decision was a tradeoff judgement, not asort.

In [ ]:
# ============================================================
# CELL 7 — COMPARISON FIGURES
# ============================================================
COLORS = {"M1": "#999999", "M2": "#1f77b4", "M3": "#2ca02c", "M4": "#d62728"}
BAR_METRICS = [("accuracy", "Accuracy"), ("f1_macro", "Macro-F1"),
               ("recall_macro", "Sensitivity"), ("specificity_macro", "Specificity"),
               ("icbhi_score", "ICBHI Score")]

# ---- (1) Grouped metric comparison ----
fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(BAR_METRICS))
width = 0.8 / len(ORDER)
for i, mid in enumerate(ORDER):
    r = ALL_MODELS[mid]
    vals = [r.get(k) or 0 for k, _ in BAR_METRICS]
    label = f"{mid}" + (" (ref)" if mid in CFG["reference_only"] else "")
    if mid == WINNER:
        label += "  ** SELECTED **"
    ax.bar(x + i * width - 0.4 + width / 2, vals, width,
           label=label, color=COLORS.get(mid, "#777"),
           edgecolor="black" if mid == WINNER else "none",
           linewidth=2 if mid == WINNER else 0,
           alpha=0.55 if mid in CFG["reference_only"] else 0.95)

ax.set_xticks(x)
ax.set_xticklabels([lbl for _, lbl in BAR_METRICS])
ax.set_ylabel("Score")
ax.set_ylim(0, 1.0)
ax.set_title("M12 — Backbone Comparison (ICBHI 2017, official 60/40 split)", fontsize=13)
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
p1 = os.path.join(CFG["out_dir"], "backbone_comparison.png")
plt.savefig(p1, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {p1}")

# ---- (2) Efficiency vs performance — the tradeoff figure ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for mid in ORDER:
    r = ALL_MODELS[mid]
    is_ref = mid in CFG["reference_only"]
    marker = "*" if mid == WINNER else ("s" if is_ref else "o")
    size = 520 if mid == WINNER else 190
    for ax, xkey, xlabel in [
            (axes[0], "model_size_mb", "Model size (MB, log scale)"),
            (axes[1], "inference_time_ms_per_sample", "Inference latency (ms/sample, log scale)")]:
        xv = r.get(xkey)
        if xv is None:
            continue
        ax.scatter(xv, r["icbhi_score"], s=size, marker=marker,
                   color=COLORS.get(mid, "#777"),
                   edgecolors="black", linewidths=1.4,
                   alpha=0.55 if is_ref else 0.95, zorder=3)
        ax.annotate(f"{mid}" + (" (selected)" if mid == WINNER else ""),
                    (xv, r["icbhi_score"]), textcoords="offset points",
                    xytext=(10, 8), fontsize=10,
                    fontweight="bold" if mid == WINNER else "normal")

for ax, xlabel in [(axes[0], "Model size (MB, log scale)"),
                   (axes[1], "Inference latency (ms/sample, log scale)")]:
    ax.set_xscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("ICBHI Score")
    ax.grid(True, alpha=0.3, which="both")

axes[0].set_title("Accuracy vs. model size")
axes[1].set_title("Accuracy vs. inference latency")
plt.suptitle("M12 — Efficiency vs. Performance Tradeoff (star = selected backbone)", fontsize=13)
plt.tight_layout()
p2 = os.path.join(CFG["out_dir"], "efficiency_vs_performance.png")
plt.savefig(p2, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {p2}")

---## Section 6 — Verify the Frozen Checkpoint`Model_Training_Reference.md:244` says the deliverable is *"one frozen checkpoint + a short writtenjustification"*. A backbone decision is only as trustworthy as the checkpoint it hands over, so thissection re-loads the winner's `best_model.pth` and confirms it reproduces the metrics the decisionwas made on.This is the check that catches the failure mode where the table says one thing and the file Member Bactually receives is something else. It needs the ICBHI dataset and the winner's checkpoint; wheneither is missing it **skips cleanly** rather than failing the notebook, and records that it wasskipped in the audit.

In [ ]:
# ============================================================
# CELL 8 — CHECKPOINT VERIFICATION (optional, self-skipping)
# ============================================================
VERIFICATION = {"attempted": False, "performed": False, "skipped_reason": None}


def find_winner_checkpoint(model_id):
    """Locate the winner's best_model.pth across repo / Drive / Kaggle layouts."""
    roots = []
    if REPO_ROOT:
        for member in ["Asif's", "Barshon's", "Farhana's", "Sami's"]:
            roots += [os.path.join(REPO_ROOT, member, model_id),
                      os.path.join(REPO_ROOT, member, model_id, "checkpoints")]
    roots += [f"/content/drive/MyDrive/OWMTL/{model_id}/checkpoints",
              f"/content/drive/MyDrive/OWMTL/{model_id}", "/content", "."]
    for r in roots:
        p = os.path.join(r, "best_model.pth")
        if os.path.exists(p):
            return p
    return None


if not CFG["verify_checkpoint"]:
    VERIFICATION["skipped_reason"] = "verify_checkpoint=False in CFG"
    print(f"Verification skipped ({VERIFICATION['skipped_reason']}).")
else:
    VERIFICATION["attempted"] = True
    ckpt_path = find_winner_checkpoint(WINNER)

    if ckpt_path is None:
        VERIFICATION["skipped_reason"] = f"no best_model.pth found for {WINNER}"
        print(f"Verification skipped: {VERIFICATION['skipped_reason']}.")
    else:
        print(f"Found {WINNER} checkpoint: {ckpt_path}")
        print(f"  file size: {os.path.getsize(ckpt_path) / 1024**2:.2f} MB "
              f"(includes optimiser state; the protocol's model_size_mb is state_dict only)")
        try:
            import torch
            DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            # §11.A — our own trusted checkpoint carrying non-tensor metadata
            state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

            print(f"  checkpoint epoch      : {state.get('epoch')}")
            print(f"  checkpoint best_score : {state.get('best_score')}")
            print(f"  model_config          : {state.get('model_config')}")

            reported_epoch = ALL_MODELS[WINNER].get("best_epoch")
            reported_score = ALL_MODELS[WINNER].get("icbhi_score")
            ck_score = state.get("best_score")

            checks = []
            if ck_score is not None and reported_score is not None:
                agree = abs(float(ck_score) - float(reported_score)) < 5e-3
                checks.append(("best_score matches results JSON", agree,
                               f"checkpoint {float(ck_score):.4f} vs reported "
                               f"{float(reported_score):.4f}"))
            n_tensors = len(state.get("model_state", {}))
            checks.append(("model_state present", n_tensors > 0, f"{n_tensors} tensors"))

            # state_dict includes non-learnable buffers (BatchNorm running_mean /
            # running_var / num_batches_tracked) that model.parameters() excludes, so a
            # small positive delta is expected and correct -- only a large gap indicates
            # the checkpoint is a different architecture than the table claims.
            sd = state.get("model_state", {})
            sd_params = int(sum(v.numel() for v in sd.values() if hasattr(v, "numel")))
            rep_params = ALL_MODELS[WINNER].get("total_params")
            if rep_params:
                delta = sd_params - rep_params
                agree_p = abs(delta) / rep_params < 0.01
                detail = (f"checkpoint state_dict {sd_params:,} vs reported params "
                          f"{rep_params:,} (+{delta:,} = BatchNorm buffers, expected)"
                          if 0 <= delta else
                          f"checkpoint state_dict {sd_params:,} vs reported params "
                          f"{rep_params:,} ({delta:,})")
                checks.append(("parameter count consistent with results JSON", agree_p, detail))

            print("\n  Verification:")
            for label, ok, detail in checks:
                print(f"    [{'x' if ok else ' '}] {label}  ({detail})")

            VERIFICATION["performed"] = True
            VERIFICATION["checkpoint_path"] = ckpt_path
            VERIFICATION["checkpoint_epoch"] = state.get("epoch")
            VERIFICATION["checkpoint_best_score"] = (
                float(ck_score) if ck_score is not None else None)
            VERIFICATION["state_dict_param_count"] = sd_params
            VERIFICATION["checks"] = [
                {"check": c[0], "passed": bool(c[1]), "detail": c[2]} for c in checks]
            VERIFICATION["all_passed"] = all(c[1] for c in checks)

            print(f"\n  Overall: {'ALL CHECKS PASSED' if VERIFICATION['all_passed'] else 'SOME CHECKS FAILED'}")
            if not VERIFICATION["all_passed"]:
                print("  The frozen checkpoint does not match the reported table. "
                      "Resolve this BEFORE handing it to Member B.")

        except ImportError:
            VERIFICATION["skipped_reason"] = "PyTorch not installed in this environment"
            print(f"  Verification skipped: {VERIFICATION['skipped_reason']}.")
        except Exception as e:
            VERIFICATION["skipped_reason"] = f"could not load checkpoint: {e}"
            print(f"  Verification skipped: {VERIFICATION['skipped_reason']}")

---## Section 7 — Deliverables`results_M12.json` (§4 schema, inheriting the winner's metrics as M12 trains nothing),the audit trail, and the written justification — the last generated **from the loaded numbers**rather than typed by hand, so it cannot contradict the table it summarises.

In [ ]:
# ============================================================
# CELL 9 — EXPORT results_M12.json (§4 + §4.1 SCHEMA)
# ============================================================
w = ALL_MODELS[WINNER]
w_raw = w.get("_raw", {})

results = {
    "meta": {
        "model_id": "M12",
        "model_name": "Backbone Selection (Final)",
        "member": "A",
        "member_name": CFG["member_name"],
        "date_completed": datetime.date.today().isoformat(),
        "is_augmented": False,
        "augmentation_method": "none",
        "notes": (
            f"M12 is a DECISION, not a new trained model — no training was performed. "
            f"Selected backbone: {WINNER} ({w['model_name']}, {w['architecture']}). "
            f"Candidates evaluated: {', '.join(CFG['candidates'])} "
            f"(Model_Training_Reference.md:114). M1 was reported for reference but excluded "
            f"from the decision, since :144 designates it a throwaway checkpoint explicitly "
            f"superseded by M2. Selection followed a 5-step rule fixed before the numbers were "
            f"read (see the 'decision' block and decision_audit.json). All metrics below are "
            f"inherited from {WINNER}."
        ),
    },

    "config": w_raw.get("config", {"architecture": w["architecture"],
                                   "inherited_from": WINNER, "seed": 42}),
    "environment": w_raw.get("environment", {"platform": "analysis-only (no training)"}),
    "dataset_info": w_raw.get("dataset_info", {
        "dataset": "ICBHI_2017", "split_method": "patient_independent_official_60_40"}),

    "efficiency": {
        "total_params": w["total_params"],
        "trainable_params": w_raw.get("efficiency", {}).get("trainable_params", w["total_params"]),
        "model_size_mb": w["model_size_mb"],
        "training_time_total_s": w_raw.get("efficiency", {}).get("training_time_total_s", 0),
        "training_time_per_epoch_s_avg":
            w_raw.get("efficiency", {}).get("training_time_per_epoch_s_avg", 0),
        "gpu_name": w_raw.get("efficiency", {}).get("gpu_name", "n/a (no training in M12)"),
        "inference_time_ms_per_sample": w["inference_time_ms_per_sample"],
        "note": f"Inherited from {WINNER}; M12 performs no training of its own.",
    },

    "best_epoch": {
        "epoch": w["best_epoch"],
        "primary_metric": "icbhi_score",
        "primary_metric_value": w["icbhi_score"],
    },

    "best_metrics": w_raw.get("best_metrics", {
        "accuracy": w["accuracy"], "precision_macro": w["precision_macro"],
        "recall_macro": w["recall_macro"], "f1_macro": w["f1_macro"],
        "specificity_macro": w["specificity_macro"], "icbhi_score": w["icbhi_score"],
    }),

    "ablation": {
        "ablation_group": "backbone_architecture",
        "ablation_role": "baseline",       # M12 IS the reference the group is measured against
        "baseline_model_id": None,
        "variable_changed": f"backbone: {w['architecture']} selected as the frozen shared encoder",
        "variables_held_constant": [
            "loss_function: inverse_frequency_class_weighted_CrossEntropyLoss",
            "data_split: patient_independent_official_60_40",
            "augmentation: none",
            "seed: 42",
            "task: 4_class_sound_event",
        ],
        "component_flags": {
            "has_sound_event_head": True, "has_disease_head": False,
            "has_cross_task_consistency": False, "has_cqkd_regularization": False,
            "has_openmax_rejection": False, "owl_stage": 0, "compression_clusters": None,
        },
        "loss_weights": {"sound_event_weight": 1.0, "disease_weight": None,
                         "consistency_weight": None},
    },

    # ── M12-specific block: the decision itself ──
    "decision": {
        "selected_model_id": WINNER,
        "selected_model_name": w["model_name"],
        "selected_architecture": w["architecture"],
        "candidates_evaluated": CFG["candidates"],
        "excluded_from_decision": {
            "M1": ("Model_Training_Reference.md:144 designates M1 a throwaway/reference "
                   "checkpoint whose numbers are explicitly superseded by M2."),
        },
        "rule_version": AUDIT["rule_version"],
        "sanity_gate_passed": AUDIT["sanity_gate_passed"],
        "audit": AUDIT,
        "comparison_table": {
            mid: {k: ALL_MODELS[mid].get(k) for k in
                  ["architecture", "accuracy", "precision_macro", "recall_macro",
                   "f1_macro", "specificity_macro", "icbhi_score", "total_params",
                   "model_size_mb", "inference_time_ms_per_sample", "source"]}
            for mid in ORDER
        },
        "checkpoint_verification": VERIFICATION,
        "robustness_m1_counterfactual": ROBUSTNESS,
    },

    "training_history": w_raw.get("training_history", []),
}

out_json = os.path.join(CFG["out_dir"], "results_M12.json")
with open(out_json, "w") as f:
    json.dump(results, f, indent=2, cls=NumpyEncoder)
print(f"Saved: {out_json}")

out_audit = os.path.join(CFG["out_dir"], "decision_audit.json")
with open(out_audit, "w") as f:
    json.dump({"audit": AUDIT, "candidates": {m: {k: v for k, v in ALL_MODELS[m].items()
                                                  if k != "_raw"} for m in ORDER}},
              f, indent=2, cls=NumpyEncoder)
print(f"Saved: {out_audit}")

In [ ]:
# ============================================================
# CELL 10 — GENERATE THE WRITTEN JUSTIFICATION (§244 deliverable)
# ============================================================
# Written from the loaded numbers, never typed by hand. The previous M12 justification
# drifted from its own data (it declared M1 the winner in the header while arguing for
# AST in three of four rationale sections, quoting M1's parameter count as AST's).
# Generating it removes that failure mode entirely.

sep_step = next(s for s in AUDIT["steps"] if s["name"] == "statistical_separation")
tie_step = next(s for s in AUDIT["steps"] if s["name"] == "efficiency_tiebreak")

lines = []
A = lines.append

A(f"# M12 — Backbone Selection Justification")
A("")
A(f"**Decision:** `{WINNER}` — {w['model_name']} (`{w['architecture']}`) is selected as the "
  f"final frozen shared backbone.")
A("")
A(f"**Author:** {CFG['member_name']} (Member A) &nbsp;|&nbsp; "
  f"**Date:** {datetime.date.today().isoformat()} &nbsp;|&nbsp; "
  f"**Rule version:** {AUDIT['rule_version']}")
A("")
A("> Generated programmatically from the candidates' `results_M*.json` files by "
  "`M12_backbone_selection.ipynb`. Every number below is read from those files, so this "
  "document cannot drift from the data it describes.")
A("")
A("---")
A("")
A("## 1. Candidates")
A("")
A("`Model_Training_Reference.md:114` requires M2, M3 and M4 as the inputs to this decision.")
A("")
A("| Model | Architecture | Acc | Macro-F1 | Se | Sp | ICBHI | Params | Size (MB) | ms/sample |")
A("|---|---|---|---|---|---|---|---|---|---|")
for mid in ORDER:
    r = ALL_MODELS[mid]
    tag = " *(reference only)*" if mid in CFG["reference_only"] else ""
    star = " **← selected**" if mid == WINNER else ""
    A(f"| **{mid}**{tag}{star} | `{r['architecture']}` | {fmt(r['accuracy'])} | "
      f"{fmt(r['f1_macro'])} | {fmt(r['recall_macro'])} | {fmt(r['specificity_macro'])} | "
      f"{fmt(r['icbhi_score'])} | {(r['total_params'] or 0):,} | "
      f"{fmt(r['model_size_mb'], '.2f')} | {fmt(r['inference_time_ms_per_sample'], '.2f')} |")
A("")
A("All rows share the same evaluation setup — ICBHI 2017, official patient-independent 60/40 "
  "split, cycle-level, 4-class sound event, inverse-frequency class-weighted "
  "`CrossEntropyLoss`, no augmentation, seed 42 — so the comparison is free of the "
  "loss-formulation confound `Model_Training_Reference.md:243` warns against.")
A("")
A("### Why M1 is excluded")
A("")
A("`Model_Training_Reference.md:144` is explicit that M1 is a throwaway: *\"treat this as a "
  "throwaway/reference checkpoint — the real reported numbers come from M2 (CNN baseline, "
  "tuned).\"* It is reported above for context and excluded from the decision. "
  f"In any case {WINNER} dominates it on every reported metric, so its exclusion does not "
  "change the outcome.")
A("")
A("---")
A("")
A("## 2. Decision rule")
A("")
A("`Model_Training_Reference.md:243` requires *\"a deliberate, documented efficiency-vs-accuracy "
  "tradeoff decision, not just 'highest accuracy wins'\"*. The rule below was fixed in code "
  "before the numbers were read:")
A("")
A("1. **Eligibility** — candidates are M2, M3, M4; M1 excluded.")
A("2. **Rank** by ICBHI score, the field-standard primary metric.")
A("3. **Separation** — the leader wins outright only if its margin over the runner-up exceeds "
  "the leader's own cross-validation standard deviation, measured across patient-grouped folds.")
A("4. **Efficiency tiebreak** — among statistically tied candidates, prefer the more "
  "deployable one (params → size → latency).")
A("5. **Sanity gate** — the winner must not lose to another candidate on *both* accuracy "
  "and macro-F1.")
A("")
A("---")
A("")
A("## 3. How the rule resolved")
A("")
A(f"**Ranking by ICBHI:** " + " > ".join(
    f"{r['model']} ({r['icbhi_score']:.4f})"
    for r in next(s for s in AUDIT['steps'] if s['name'] == 'rank_by_primary_metric')['ranking']))
A("")
A(f"**Separation test:** {sep_step['leader']} leads {sep_step['runner_up']} by "
  f"**{sep_step['margin']:.4f}** ICBHI. The tolerance is {sep_step['leader']}'s "
  f"cross-validation std of **{sep_step['leader_cv_std']:.4f}** "
  f"(measured across patient-grouped folds in its own hyperparameter sweep).")
A("")
if sep_step["separated"]:
    A(f"Because {sep_step['margin']:.4f} > {sep_step['threshold']:.4f}, the lead is larger than "
      f"the fold-to-fold noise of the architecture itself. **{sep_step['leader']} is "
      f"statistically separated and wins outright** — the efficiency tiebreak was not needed.")
else:
    A(f"Because {sep_step['margin']:.4f} ≤ {sep_step['threshold']:.4f}, the candidates "
      f"{', '.join(tie_step.get('tied_candidates', []))} are statistically indistinguishable. "
      f"The efficiency tiebreak therefore applied and selected **{tie_step.get('selected')}** "
      f"as the most deployable of the tied group.")
A("")
A(f"**Sanity gate:** {'PASSED' if AUDIT['sanity_gate_passed'] else 'FAILED'} — "
  + (f"no other candidate beats {WINNER} on both accuracy and macro-F1."
     if AUDIT['sanity_gate_passed'] else
     f"candidate(s) {AUDIT['steps'][4]['models_beating_winner_on_all_secondaries']} beat "
     f"{WINNER} on both secondary metrics; review before freezing."))
A("")
A("---")
A("")
A("### Robustness: does the decision depend on excluding M1?")
A("")
A(f"The same rule was re-run with M1 promoted to a full candidate. "
  f"It would select **{ROBUSTNESS['counterfactual_winner']}**, and that selection "
  f"{'passes' if ROBUSTNESS['counterfactual_sanity_gate_passed'] else 'FAILS'} the sanity gate.")
A("")
A(ROBUSTNESS["conclusion"])
A("")
A("---")
A("")
A("## 4. Efficiency and downstream fit")
A("")
_m4 = ALL_MODELS.get("M4", {})
if _m4.get("total_params") and w.get("total_params"):
    A(f"The selected backbone is **{_m4['total_params'] / w['total_params']:.1f}× smaller** "
      f"in parameters than the AST candidate (M4) "
      f"({w['total_params']:,} vs {_m4['total_params']:,}) and "
      f"**{(_m4['inference_time_ms_per_sample'] or 1) / (w['inference_time_ms_per_sample'] or 1):.1f}× "
      f"faster** at inference "
      f"({w['inference_time_ms_per_sample']} vs {_m4['inference_time_ms_per_sample']} ms/sample), "
      f"while scoring **{w['icbhi_score'] - _m4['icbhi_score']:+.4f}** ICBHI against it.")
    A("")
A(f"At {w['total_params']:,} parameters / {w['model_size_mb']} MB this sits in a sensible range "
  "for Member C's CQKD compression workstream (M16/M18): large enough to be a meaningful teacher, "
  "small enough that the compressed student remains a credible deployability claim. "
  "`Model_Training_Reference.md:243` names that downstream constraint as an explicit input to "
  "this decision.")
A("")
A("---")
A("")
A("## 5. Frozen checkpoint")
A("")
if VERIFICATION.get("performed"):
    A(f"The handoff checkpoint was re-loaded and checked against the reported table:")
    A("")
    for c in VERIFICATION.get("checks", []):
        A(f"- {'✅' if c['passed'] else '❌'} {c['check']} — {c['detail']}")
    A("")
    A(f"**Overall: {'all checks passed' if VERIFICATION.get('all_passed') else 'SOME CHECKS FAILED'}.**"
      + ("" if VERIFICATION.get("all_passed") else
         " Do not hand this checkpoint downstream until resolved."))
    A("")
    A(f"Path: `{VERIFICATION.get('checkpoint_path')}`")
else:
    A(f"Checkpoint verification was not performed "
      f"({VERIFICATION.get('skipped_reason') or 'not attempted'}). "
      f"Re-run this notebook with the {WINNER} checkpoint available to complete this section.")
A("")
A("---")
A("")
A("## 6. Consequences for downstream models")
A("")
A(f"Every downstream model that consumes \"the M12 backbone\" — M13 (disease head), "
  f"M15 (cross-task consistency), M17 (OWL Stage 2), and Member C's M16/M18 distillation — "
  f"should be built on **{WINNER}**.")
A("")
if prev in ALL_MODELS and WINNER != prev:
    A(f"An earlier pass at M12 selected **{prev}**, made when M2 and M3 did not yet exist and "
      f"only two models could be compared. {WINNER} improves on {prev} by "
      f"**{w['icbhi_score'] - ALL_MODELS[prev]['icbhi_score']:+.4f}** ICBHI, "
      f"**{w['accuracy'] - ALL_MODELS[prev]['accuracy']:+.4f}** accuracy and "
      f"**{w['f1_macro'] - ALL_MODELS[prev]['f1_macro']:+.4f}** macro-F1. Any downstream model "
      f"already trained against {prev} is therefore sitting on a strictly weaker encoder and "
      f"should be re-fit against {WINNER} before its numbers go in the paper.")
A("")
A("---")
A("")
A("## 7. Reproducibility")
A("")
A("- `results_M12.json` — §4-schema record including the full `decision` block")
A("- `decision_audit.json` — every rule step with its inputs and outcome")
A("- `backbone_comparison.png`, `efficiency_vs_performance.png` — the figures above")
A("")
A("Sources for each row:")
A("")
for mid in ORDER:
    A(f"- **{mid}**: {ALL_MODELS[mid]['source']}")

justification = "\n".join(lines)
out_md = os.path.join(CFG["out_dir"], "M12_backbone_justification.md")
with open(out_md, "w") as f:
    f.write(justification + "\n")

print(f"Saved: {out_md}\n")
print("=" * 78)
print(justification[:2600])
print("...")
print("=" * 78)

In [ ]:
# ============================================================
# FINAL CELL — TEAM HANDOFF & ONE-CLICK FILE DOWNLOADS (§11.D)
# ============================================================
import shutil, glob
try:
    from IPython.display import display, FileLink
    HAS_IPY = True
except ImportError:
    HAS_IPY = False

print("=" * 66)
print("M12 DELIVERABLES")
print("=" * 66)

protocol_files = sorted(
    glob.glob(os.path.join(CFG["out_dir"], "results_M12.json")) +
    glob.glob(os.path.join(CFG["out_dir"], "decision_audit.json")) +
    glob.glob(os.path.join(CFG["out_dir"], "M12_backbone_justification.md")) +
    glob.glob(os.path.join(CFG["out_dir"], "*.png"))
)

for fpath in protocol_files:
    size_mb = round(os.path.getsize(fpath) / (1024 ** 2), 3)
    print(f"Ready: {os.path.basename(fpath):<34} ({size_mb} MB)")
    if HAS_IPY:
        display(FileLink(fpath))

if protocol_files:
    bundle_dir = os.path.join(CFG["out_dir"], "protocol_bundle")
    os.makedirs(bundle_dir, exist_ok=True)
    for fpath in protocol_files:
        shutil.copy2(fpath, os.path.join(bundle_dir, os.path.basename(fpath)))
    zip_path = shutil.make_archive(os.path.join(CFG["out_dir"], "M12_handoff_bundle"),
                                   "zip", bundle_dir)
    print(f"\nZIP bundle: {zip_path} "
          f"({round(os.path.getsize(zip_path) / (1024 ** 2), 2)} MB)")
    if HAS_IPY:
        display(FileLink(zip_path))

print("=" * 66)
print(f"\nDECISION: {WINNER} is the frozen shared backbone.")
print(f"\nSend to Member B (M13/M15/M17) and Member C (M16/M18):")
print(f"  1. {WINNER}'s best_model.pth   — the frozen encoder to build on")
print(f"  2. M12_backbone_justification.md — why, with the numbers")
print(f"  3. results_M12.json              — the M28 merge row")
print("=" * 66)

---## Section 8 — Summary & Next Steps### What M12 establishesThe frozen shared backbone, chosen by a rule fixed before the numbers were read, from the completecandidate set the reference requires. The justification document, the audit trail, and the §4results JSON are all generated from the same loaded data, so they cannot disagree with each other.### What to do with it1. **Commit** `results_M12.json`, `M12_backbone_justification.md`, `decision_audit.json` and both   PNGs into `Asif's/M12/`.2. **Tell Members B and C.** They consume "the M12 backbone" directly — M13 → M15 → M17 for B,   M16/M18 for C. If the selected backbone differs from what they have been building against, that   is a re-fit, and the sooner they know the cheaper it is.3. **Supersede the old justification.** `Barshon's/M12/M12_backbone_justification.txt` was written   against a two-candidate comparison and states a different winner; leaving both in the repo   invites the wrong one being cited.### Honest caveats to carry into the write-up- **Best-checkpoint selection used the test split** for every candidate (M1–M4 alike). The  comparison between them is fair because they all did the same thing, but no absolute number here  is a clean held-out estimate. The k-fold spreads in `M2_hp_sweep.json` / `M3_hp_sweep.json` are  the honest generalisation figures.- **M4's row is transcribed, not loaded.** Its `results_M4.json` is not in the repo. Ask Barshon to  commit it, then re-run this notebook — it will pick the file up automatically and replace the  transcription.- **The AST result is well below its literature reputation.** M4 scoring last is a real finding  worth a sentence in the paper, but it is also worth checking it was given a fair shot (it trained  40 epochs with the best checkpoint at epoch 10, suggesting it overfit early rather than  undertrained).